In [ ]:
import demucs.api
import soundfile as sf
import torch
import IPython.display as ipd
import numpy

In [ ]:
print("1. Caricamento del modello Demucs in corso...")
separator = demucs.api.Separator()

print("2. Leggo il file...")
# Leggiamo l'audio nativamente con soundfile
audio_data, sample_rate = sf.read("songs/test.mp3")

# Trasformiamo l'array NumPy in un Tensore PyTorch
# Attenzione: soundfile legge [Tempo, Canali], ma Demucs vuole [Canali, Tempo]. Usiamo .T per trasporre la matrice!
wav_tensor = torch.tensor(audio_data).T.float()

# Se l'audio ha un solo canale (Mono), Demucs andrebbe in errore perché si aspetta audio Stereo.
# Quindi lo sdoppiamo su due canali:
if wav_tensor.ndim == 1:
    wav_tensor = wav_tensor.unsqueeze(0).repeat(2, 1)

print("3. Separazione audio iniziata. Potrebbe volerci qualche minuto...")
origin, separated = separator.separate_tensor(wav_tensor, sample_rate)

print("\nSeparazione completata! Tracce estratte:", list(separated.keys()))

print("\nInizio il salvataggio delle tracce...")

# Usiamo .items() per prendere sia il nome della traccia che il suo audio
for track_name, track_tensor in separated.items():
    
    # 1. Spostiamo l'audio su CPU, convertiamo in NumPy e rigiriamo la matrice con .T
    audio_array = track_tensor.cpu().numpy().T
    
    # 2. Creiamo il nome del file (es: "vocals.wav", "drums.wav")
    file_name = f"output/test_{track_name}.wav"
    
    # 3. Salviamo fisicamente il file sul PC
    sf.write(file_name, audio_array, sample_rate)
    print(f"Salvato con successo: {file_name}")
    
    # 4. (Opzionale) Creiamo il player audio direttamente in Jupyter per ascoltarla subito
    #player = ipd.Audio(track_tensor.cpu().numpy(), rate=sample_rate)
    #display(player)
    #commentata perchè espode la dimensione del file!!

print("\nFinito! Hai separato la tua prima traccia con Demucs!")


In [ ]:
print(torch.cuda.is_available())